<a href="https://colab.research.google.com/github/MiguelCortezPino/etl-data-pipeline/blob/main/notebook/Tipos_seguro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd

In [12]:
url = "https://raw.githubusercontent.com/MiguelCortezPino/etl-data-pipeline/refs/heads/main/data/raw/tipos_seguro.csv"

df = pd.read_csv(url)

df.head()


,id_tipo_seguro,tipo,categoria,riesgo_base
0,1,Pyme,Familiar,-
1,2,Industrial,Empresarial,4.68
2,3,Industrial,Familiar,5.10
3,4,Industrial,Personal,NaN
4,5,Auto,empresarial,9.07


In [13]:
#Exploracion de datos
df.shape
df.columns
df.info()
df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 4 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   id_tipo_seguro  12 non-null     int64 
 1   tipo            12 non-null     object
 2   categoria       10 non-null     object
 3   riesgo_base     10 non-null     object
dtypes: int64(1), object(3)
memory usage: 516.0+ bytes


,0
id_tipo_seguro,0
tipo,0
categoria,2
riesgo_base,2


In [14]:
#Limpieza de datos
tipos_seguro = df.copy()
print(tipos_seguro['categoria'].value_counts(dropna=False))

categoria
empresarial     3
Familiar        2
Empresarial     2
Personal        2
NaN             2
Especial        1
Name: count, dtype: int64


In [15]:
tipos_seguro['categoria'] = tipos_seguro['categoria'].str.strip().str.title()
print(tipos_seguro['categoria'].value_counts(dropna=False))

categoria
Empresarial    5
Familiar       2
Personal       2
NaN            2
Especial       1
Name: count, dtype: int64


In [16]:
tipos_seguro = df.copy()
print(tipos_seguro['tipo'].value_counts(dropna=False))

tipo
Industrial    4
Auto          2
Pyme          1
Salud         1
Educación     1
Accidentes    1
Dental        1
Agrícola      1
Name: count, dtype: int64


In [17]:
tipos_seguro = df.copy()
print(tipos_seguro['riesgo_base'].value_counts(dropna=False))

riesgo_base
NaN       2
 4.68     1
-         1
5.10      1
9.07      1
2.52      1
 0.92     1
 7.42     1
 5.68     1
2.70      1
4.33      1
Name: count, dtype: int64


In [18]:
#Funcion para estandarizar
def limpiar_numero_mixto(v):
    if pd.isna(v): return None

    s = str(v).strip().replace('$', '').replace(' ', '')

    if s in ('', '-', 'N/A', 'nan', 'None'): return None

    if ',' in s and '.' in s:
        s = s.replace('.' if s.find('.') < s.find(',') else ',', '')

    try:
        return float(s.replace(',', '.'))
    except:
        return None

In [19]:
tipos_seguro['riesgo_base'] = tipos_seguro['riesgo_base'].apply(limpiar_numero_mixto)
print(tipos_seguro['riesgo_base'].value_counts(dropna=False))


riesgo_base
NaN     3
4.68    1
5.10    1
9.07    1
2.52    1
0.92    1
7.42    1
5.68    1
2.70    1
4.33    1
Name: count, dtype: int64


In [20]:
#Separar datos válidos y rechazados
filtrado_validos = (tipos_seguro['categoria'].notna()) & \
                   (tipos_seguro['riesgo_base'].notna())
validos = tipos_seguro[filtrado_validos].copy()
rechazados = tipos_seguro[~filtrado_validos].copy()


In [21]:
#Motivo de rechazo
def motivo_rechazo(row):
  motivos = []
  if pd.isnull(row['categoria']):
    motivos.append('categoria_vacia')
  if pd.isnull(row['riesgo_base']):
    motivos.append('riesgo_base_vacio')
  return ','.join(motivos)

  rechazados['motivo_rechazo'] = rechazados.apply(motivo_rechazo, axis=1)


In [22]:
validos.to_csv("tipos_seguro_curated.csv", index=False)

rechazados.to_csv("tipos_seguro_rejects.csv", index=False)